# param-grad-access — worked example 3: classify each grad as nan / zero / ok

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `param-grad-access`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Inspecting `.grad` per parameter classifies training health: `nan` if any NaN or Inf, `zero` if all finite and entirely zero, else `ok`. The NaN/Inf check MUST come first, since an all-NaN tensor would confuse the zero check. Params with `.grad is None` are omitted.

## Worked solution

We implement `classify_grad_health(model)`. For each named parameter we skip `None`, then check NaN/Inf FIRST (order matters because NaN compares unequal to both zero and nonzero), label `zero` if the max absolute value is exactly zero, and otherwise `ok`. We build a model, manually plant one NaN gradient, one all-zero gradient, and one ordinary gradient, then run the classifier. We print the resulting label per parameter to show the three buckets are correctly assigned.

In [ ]:
import torch as t
import torch.nn as nn

t.manual_seed(2)

def classify_grad_health(model):
    out = {}
    for name, p in model.named_parameters():
        if p.grad is None:
            continue
        if t.isnan(p.grad).any().item() or t.isinf(p.grad).any().item():
            out[name] = 'nan'
        elif p.grad.abs().max().item() == 0.0:
            out[name] = 'zero'
        else:
            out[name] = 'ok'
    return out

model = nn.Sequential(nn.Linear(2, 2), nn.Linear(2, 2))
params = list(model.parameters())
params[0].grad = t.tensor([[float('nan'), 1.0], [0.0, 0.0]])
params[1].grad = t.zeros(2)
params[2].grad = t.ones(2, 2)
params[3].grad = t.zeros(2)
print(classify_grad_health(model))